In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd


import os, re, warnings
import numpy as np
import pandas as pd
import mne
from scipy.sparse import csr_matrix


import pickle
from scipy.sparse import triu

from mne.channels.layout import _find_topomap_coords

In [2]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
modality="visual"
layer_script = "event"
subj= "s01b"
type_epoch = "emoc"


# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, modality=modality,layer_script=layer_script,  subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")
    
    
    

#modify this only if you want to use dynamic analysis, or static event analysis
if layer_script=="event":
    dynamic=False
    
elif layer_script=="block":
    dynamic=False # it will ALWAYS be false in block



    

✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\ICA_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_clean_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_matlab_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\evoked_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\channels_structure
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\raw_hsp
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\fwd
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\inverse
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_event\acw_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_eve

In [4]:
filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")

Filtrado aplicado: 1-40 Hz


# import CSV

In [5]:
if filtering:
    input_path = ACW_path / f"table_merged_ACW_FOOOF_{filter_name}_{layer_script}_{type_epoch}.csv"
    
    if dynamic:
        input_path = ACW_path / f"table_merged_dynamic_ACW_FOOOF_results_{filter_name}_{layer_script}_{type_epoch}.csv"
        
else:
    input_path = ACW_path / f"table_merged_df_{layer_script}_{type_epoch}.csv"

if not input_path.exists():
    raise FileNotFoundError(f"No existe el archivo: {input_path}")

table_merged_cleaned_df = pd.read_csv(input_path)

print(f"Loaded from {input_path}")
print(f"Shape: {table_merged_cleaned_df.shape}")

Loaded from g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_event\acw_event\table_merged_ACW_FOOOF_filt_1-40_event_emoc.csv
Shape: (367688, 21)


In [6]:
table_merged_cleaned_df.columns

Index(['Subject', 'Condition_self', 'Condition_emotion', 'Condition_gaze',
       'Epoch', 'Epoch_relative', 'Channel', 'acw_50_elect_all_epoch_all',
       'acw_0_elect_all_epoch_all', 'delta', 'theta', 'alpha', 'beta', 'gamma',
       'offsets', 'exponents', 'r2', 'error', 'event_id_x', 'event_id_y',
       '_merge'],
      dtype='object')

In [12]:

df = table_merged_cleaned_df.copy()

df = df.rename(columns={
    "acw_50_elect_all_epoch_all": "ACW_50", 
    "acw_0_elect_all_epoch_all": "ACW_0", 
    
})

# Columns that define the final repeated-measures structure
# Do not include epoch or condition_gaze because we want to average over them
group_cols = [
    "Subject",
    "Condition_self",
    "Condition_emotion",
    "Channel"  # change this if your channel column has another name
]

# Candidate oscillatory power columns
band_keywords = [
    "delta",
    "theta",
    "alpha",
    "beta",
    "gamma"
]

band_cols = [
    col for col in df.columns
    if any(band in col.lower() for band in band_keywords)
    and "exponent" not in col.lower()
]

# Exponent columns should be kept as they are
exponent_cols = [
    col for col in df.columns
    if "exponent" in col.lower()
]

# Core dependent variable
acw_cols = ["ACW_50"]

# Columns to average over epoch and condition_gaze
value_cols = acw_cols + band_cols + exponent_cols

# Keep only columns that exist and avoid duplicates
value_cols = list(dict.fromkeys([col for col in value_cols if col in df.columns]))

# Average over epoch and condition_gaze
df_avg = (
    df
    .groupby(group_cols, as_index=False)[value_cols]
    .mean()
)

# Z-normalize oscillatory band columns only
# Exponent columns are left unchanged
for col in band_cols:
    if col in df_avg.columns:
        mean = df_avg[col].mean()
        std = df_avg[col].std(ddof=0)
        
        if std == 0 or np.isnan(std):
            df_avg[f"{col}_z"] = np.nan
        else:
            df_avg[f"{col}_z"] = (df_avg[col] - mean) / std

print("Grouped dataframe shape:", df_avg.shape)
print("Band columns:", band_cols)
print("Exponent columns:", exponent_cols)

df_avg.head()

Grouped dataframe shape: (15930, 16)
Band columns: ['delta', 'theta', 'alpha', 'beta', 'gamma']
Exponent columns: ['exponents']


,Subject,Condition_self,Condition_emotion,Channel,ACW_50,delta,theta,alpha,beta,gamma,exponents,delta_z,theta_z,alpha_z,beta_z,gamma_z
0,s01b,friend,negative,AF3,0.019681,0.016596,0.316817,0.388649,0.593108,0.141591,1.124631,-0.965632,1.444883,-0.229593,0.751533,-0.157430
1,s01b,friend,negative,AF4,0.022085,0.023684,0.357226,0.435916,0.582589,0.073933,1.241668,-0.856936,1.799313,-0.090458,0.631422,-0.940334
2,s01b,friend,negative,AF7,0.021334,0.012868,0.198153,0.512767,0.534873,0.105810,1.188935,-1.022816,0.404087,0.135763,0.086555,-0.571469
3,s01b,friend,negative,AF8,0.019681,0.024204,0.206142,0.460597,0.504311,0.139872,1.053800,-0.848958,0.474155,-0.017806,-0.262441,-0.177323
4,s01b,friend,negative,C1,0.022386,0.077747,0.463059,0.331035,0.599930,0.039255,1.251193,-0.027762,2.727573,-0.399186,0.829437,-1.341612


In [13]:
df_avg.columns

Index(['Subject', 'Condition_self', 'Condition_emotion', 'Channel', 'ACW_50',
       'delta', 'theta', 'alpha', 'beta', 'gamma', 'exponents', 'delta_z',
       'theta_z', 'alpha_z', 'beta_z', 'gamma_z'],
      dtype='object')